# NavPlotter: addestramento del modello marino

Questo notebook addestra un YOLOv8s che riconosce **barche, navi, barche a vela, persone in acqua, boe e ostacoli
galleggianti**, partendo dal modello COCO e dai dataset marini che gli dai. Alla fine scarichi uno zip con
`best.onnx`, le immagini di calibrazione e `classes.json`: lo compili per l'Hailo con `training/compile.sh`
(vedi il README della cartella `training`).

Prima di tutto: **Runtime → Cambia tipo di runtime → GPU** (la T4 gratuita va bene). Poi esegui le celle in ordine.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics roboflow
%cd /content
# Gli script di NavPlotter: la prima volta li scarica, le volte dopo li aggiorna
!git -C Navplotter-Linux pull -q 2>/dev/null || git clone -q https://github.com/digitalimpresys/Navplotter-Linux.git

## 1. I dati

Metti tutto in `/content/inputs`, una cartella (o un file COCO `.json`) per ogni dataset:

- **I tuoi filmati**: dalla pagina Telecamera scarichi lo zip di una registrazione; etichettalo (Roboflow, CVAT o
  Label Studio, esporta in formato YOLO) e caricalo qui o su Google Drive.
- **Roboflow Universe** (universe.roboflow.com): cerca per esempio *buoy*, *boat detection*, *marine obstacle*,
  *man overboard*, *sea drone*. Dalla pagina del dataset, *Download → YOLOv8 → show download code* ti dà le righe
  da incollare nella cella qui sotto (la chiave API è la tua, gratuita).
- **SeaDronesSee** (seadronessee.cs.uni-tuebingen.de, *Object Detection v2*), **LaRS** e **MODS**: scaricali dal loro sito,
  scompattali in `/content/inputs` e indica il loro `.json` COCO.

Le classi dei dataset vengono tradotte nelle nostre con i sinonimi di `training/classes.json`; quelle che non
conosce le scarta e te le elenca, così puoi aggiungerle ai sinonimi se servono.

In [ ]:
# Facoltativo: i file da Google Drive (per esempio le tue registrazioni etichettate, come zip)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/inputs
# !unzip -q -o "/content/drive/MyDrive/navplotter/registrazione-etichettata.zip" -d /content/inputs/mie-registrazioni

In [ ]:
# Dataset di Roboflow Universe. La chiave API è gratuita: roboflow.com -> Settings -> API Keys (la "Private API Key").
# NON scriverla qui dentro: il notebook è su GitHub e chiunque la vedrebbe. Mettila nei Secret di Colab:
# icona della chiave nella barra a sinistra -> "Aggiungi nuovo secret", nome ROBOFLOW_KEY, valore la chiave,
# e attiva "Accesso al notebook". In alternativa questa cella te la chiede ogni volta.
# Per partire c'è già maritime_data (circa 4000 immagini: barche, persone, boe, navi civili e militari, CC BY 4.0).
# Altri dataset: dalla loro pagina, Download -> YOLOv8 -> show download code, e aggiungi le righe qui sotto.
from getpass import getpass
try:
    from google.colab import userdata
    ROBOFLOW_KEY = userdata.get('ROBOFLOW_KEY')
except Exception:
    ROBOFLOW_KEY = getpass('Chiave API di Roboflow (Invio per saltare): ').strip()

if ROBOFLOW_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_KEY)
    rf.workspace("jove2874").project("maritime_data").version(3).download("yolov8", location="/content/inputs/maritime_data")
    # rf.workspace("...").project("...").version(1).download("yolov8", location="/content/inputs/altro")
else:
    print("Nessuna chiave Roboflow: salto il download")

In [ ]:
import glob, os
inputs = sorted(glob.glob('/content/inputs/*/'))
# I dataset COCO: aggiungi qui i loro .json, per esempio
# inputs += ['/content/inputs/seadronessee/annotations/instances_train.json']
if not inputs:
    raise SystemExit("Nessun dataset in /content/inputs: metti la chiave Roboflow nella cella sopra, "
                     "oppure scompatta lì le tue registrazioni etichettate, e riesegui.")
print('\n'.join(inputs))
!rm -rf /content/dataset
!python3 Navplotter-Linux/training/merge_datasets.py -o /content/dataset {' '.join(inputs)}

## 2. L'addestramento

Con qualche migliaio di immagini sulla T4 ci vuole un'ora o due. `patience` lo ferma da solo quando smette di
migliorare. Se Colab ti scollega, riesegui la cella con `resume=True` sul `last.pt`.

In [ ]:
import os
from ultralytics import YOLO
assert os.path.exists('/content/dataset/data.yaml'), "Manca il dataset: esegui prima le celle della sezione 1"
model = YOLO('yolov8s.pt')           # yolov8s: è quello che il Model Zoo di Hailo sa compilare con lo script
model.train(data='/content/dataset/data.yaml', imgsz=640, epochs=100, patience=20, batch=16,
            project='/content/runs', name='marine', exist_ok=True)
# Per riprendere: YOLO('/content/runs/marine/weights/last.pt').train(resume=True)

In [ ]:
# Come se la cava sulle immagini di validazione: mAP50 per classe
metrics = YOLO('/content/runs/marine/weights/best.pt').val(data='/content/dataset/data.yaml')
for i, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f'{metrics.names[int(i)]:10s} {ap:.2f}')

In [ ]:
# Una prova a occhio su qualche immagine di validazione
import glob
from IPython.display import Image, display
res = YOLO('/content/runs/marine/weights/best.pt').predict(sorted(glob.glob('/content/dataset/images/val/*'))[:6], conf=0.3)
for r in res:
    r.save(filename='/content/pred.jpg'); display(Image('/content/pred.jpg', width=640))

## 3. Esportazione per l'Hailo

L'ONNX va compilato per l'Hailo-8 su un PC Linux x86 con la Hailo AI Software Suite (vedi README). Lo zip
contiene anche 512 immagini del dataset per la calibrazione.

In [ ]:
import random, shutil, os, glob
YOLO('/content/runs/marine/weights/best.pt').export(format='onnx', imgsz=640, opset=11)
out = '/content/navplotter-marine'
shutil.rmtree(out, ignore_errors=True); os.makedirs(out + '/calib')
shutil.copy('/content/runs/marine/weights/best.onnx', out)
shutil.copy('/content/runs/marine/weights/best.pt', out)
shutil.copy('/content/dataset/classes.json', out)
imgs = sorted(glob.glob('/content/dataset/images/train/*'))
random.seed(0)
for p in random.sample(imgs, min(512, len(imgs))):
    shutil.copy(p, out + '/calib')
shutil.make_archive('/content/navplotter-marine', 'zip', out)
from google.colab import files
files.download('/content/navplotter-marine.zip')